In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

# Libraries

In [2]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==4.53.3" \
    "peft==0.17.1" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb" \
    "evaluate"

In [3]:
%%capture
%cd /content
!git clone https://github.com/alxxtexxr/LegameX.git
%cd LegameX
!git pull

In [4]:
import os
import math
import json
from datetime import datetime
from pathlib import Path

import torch
import torch.nn as nn
from huggingface_hub import snapshot_download
from transformers import (
    AutoModelForQuestionAnswering,
    AutoModelForMaskedLM,
    DataCollatorWithPadding,
    AutoTokenizer,
)
from huggingface_hub import HfApi
from peft import LoraConfig
from safetensors.torch import load_file, save_file
from datasets import load_dataset, Dataset

# Configuration

In [5]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = 'FacebookAI/xlm-roberta-base'
REF_LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-en-2K-s42-LoRA-v260929201303'
REF_LORA_CKPT_STEP = 1000
LEGAMEX_ID = 'alxxtexxr/XLM-R-Base-squad-en-15K-s42-LegameX-LoRA-v261001202047'
LEGAMEX_CKPT_STEP = 4800

# Automatically get the username of the current user on Hugging Face Hub
user_info = HfApi().whoami()
username = user_info["name"]

# Define the hub merged model ID
_, legamex_name = LEGAMEX_ID.split("/")
legamex_name_0, _ = legamex_name.split("-v")
hub_merged_model_id = f"{username}/{legamex_name_0}-mrg-v{datetime.now().strftime('%y%m%d%H%M%S')}"
print(f"Hub merged model ID: {hub_merged_model_id}")

Hub merged model ID: alxxtexxr/XLM-R-Base-squad-en-15K-s42-LegameX-LoRA-mrg-v261004162300


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


# Utilities

In [6]:
# LoRA utilities
def download_hf_model(
        repo_id: str, 
        ckpt_step: int, 
        max_checkpoints: int = 10_000,
        ckpt_interval: int = 25,
    ) -> tuple[Path, Path | None]:
    assert len(repo_id.split("/")) == 2, \
        f"Invalid repo_id format: {repo_id}. Expected format: 'username/repo_name'."
    
    local_dir = Path(repo_id.split('/')[-1])
    ignore_checkpoints = None
    
    if ckpt_step is not None:
        ignore_checkpoints = [
            f'checkpoint-{i}/*' for i in range(0, max_checkpoints, ckpt_interval) 
            if i != ckpt_step
        ]
        ignore_checkpoints += ['runs/*', '**/wandb/*']

    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        ignore_patterns=ignore_checkpoints,
    )

    ckpt_dir = None
    if ckpt_step is not None:
        ckpt_dir = local_dir / f'checkpoint-{ckpt_step}'
    return local_dir, ckpt_dir

def check_parameter(n, p):
    print(f"{'name':<8}:", n)
    print(f"{'device':<8}:", p.device)
    print(f"{'dtype':<8}:", p.dtype)
    print(f"{'norm':<8}:", p.norm().item())

def check_weights(model, infix, limit=1):
    for i, (n, p) in enumerate(model.named_parameters()):
        if infix in n:
            check_parameter(n, p)
            print()
            if i >= limit - 1:
                break

# Data utilities
def load_train_val_datasets(
    lang, # e.g., 'en' | 'ja' | 'id'
    task, # 'wikipedia' | 'squad'
    train_size, val_size,
    token=None,
):
    # Validate that if the task is 'squad', the language must be 'en'
    if task == 'squad':
        assert lang == 'en', "SQuAD is English-only."
    
    # Define dataset configurations for each task
    data_configs = {
        'wikipedia': {
            'data_id': 'wikimedia/wikipedia',
            'data_dir': f'20231101.{lang}',
            'train_split': 'train',
            'val_split': 'train',
        },
        'squad': {
            'data_id': 'rajpurkar/squad',
            'data_dir': None,
            'train_split': 'train',
            'val_split': 'validation',
        },
    }
    
    # Validate that the specified task is supported
    assert task in data_configs, (
        f"Unsupported task: {task}. "
        f"Supported tasks: {list(data_configs.keys())}"
    )

    # Set up Hugging Face dataset configuration
    data_id = data_configs[task]['data_id']
    data_dir = data_configs[task]['data_dir']
    train_split = data_configs[task]['train_split']
    val_split = data_configs[task]['val_split']

    if train_split == val_split:
        # If the train and validation splits are the same, we need to sample from the same dataset stream
        dataset_stream = load_dataset(
            data_id,
            data_dir=data_dir,
            split=train_split,
            streaming=True,
            token=token,
        )

        train_data = []
        val_data = []

        for i, example in enumerate(dataset_stream):
            if i < train_size:
                train_data.append(example)
            elif i < train_size + val_size:
                val_data.append(example)
            else:
                break

    else:
        # If the train and validation splits are different, we can sample from each split separately
        def sample_split(split, size):
            dataset_stream = load_dataset(
                data_id,
                data_dir=data_dir,
                split=split,
                streaming=True,
            )

            data = []
            for i, example in enumerate(dataset_stream):
                if i >= size:
                    break
                data.append(example)
            return data

        train_data = sample_split(train_split, train_size)
        val_data = sample_split(val_split, val_size)

    return (
        Dataset.from_list(train_data),
        Dataset.from_list(val_data),
    )

# Model

In [7]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Define model class, task type, data collator, and label names
model_cls = AutoModelForQuestionAnswering
task_type = 'QUESTION_ANS'
data_collator = DataCollatorWithPadding(tokenizer)
label_names = ['start_positions', 'end_positions']

# Load the base model
base_model = model_cls.from_pretrained(MODEL_ID)

Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
# Load the reference LoRA configuration
_, ref_lora_ckpt_dir = download_hf_model(REF_LORA_ID, REF_LORA_CKPT_STEP)
if not ref_lora_ckpt_dir:
    raise RuntimeError(f"ref_lora_ckpt_dir is None because REF_LORA_CKPT_STEP is None or something went wrong.")
ref_lora_path = ref_lora_ckpt_dir / 'adapter_model.safetensors'
ref_lora_state_dict = load_file(ref_lora_path, device="cpu")

# Download the LegameX adapter
_, legamex_ckpt_dir = download_hf_model(LEGAMEX_ID, LEGAMEX_CKPT_STEP)
if not legamex_ckpt_dir:
    raise RuntimeError(f"legamex_ckpt_dir is None because LEGAMEX_CKPT_STEP is None or something went wrong.")
legamex_dir = legamex_ckpt_dir
print(f"LegameX adapter directory: {legamex_dir}")

legamex_state_dict = torch.load(legamex_dir / "legamex.pt", map_location="cpu")

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

LegameX adapter directory: XLM-R-Base-squad-en-15K-s42-LegameX-LoRA-v261001202047/checkpoint-4800


In [9]:
# Sanity check
for n, p in ref_lora_state_dict.items():
    n = n.replace("__DOT__", ".")
    if "layer.0.attention.output.dense.lora" not in n:
        continue
    check_parameter(n, p)
    print()

name    : base_model.model.roberta.encoder.layer.0.attention.output.dense.lora_A.weight
device  : cpu
dtype   : torch.float32
norm    : 1.7452430725097656

name    : base_model.model.roberta.encoder.layer.0.attention.output.dense.lora_B.weight
device  : cpu
dtype   : torch.float32
norm    : 0.2991487979888916



In [10]:
# Sanity check
for n, p in legamex_state_dict.items():
    n = n.replace("__DOT__", ".")
    if "layer.0.attention.output.dense.lora.ref" not in n:
        continue
    check_parameter(n, p)
    print()

name    : roberta.encoder.layer.0.attention.output.dense.lora.ref.A.weight
device  : cpu
dtype   : torch.float32
norm    : 1.7452430725097656

name    : roberta.encoder.layer.0.attention.output.dense.lora.ref.B.weight
device  : cpu
dtype   : torch.float32
norm    : 0.2991487979888916



In [11]:
class LegamexLora(nn.Module):
    def __init__(
            self, base_module, 
            
            # Reference (ref) and transfer (tfr) LoRA parameters
            ref_lora_config, tfr_lora_config,
                    
            # Gate parameters
            gate_warmup_steps, gate_scheduler_type='cosine', 
            gate_rank=1, gate_alpha=1, gate_dropout=0.0, 
            gate_bias=False, gate_use_rslora=False,
        ):
        super().__init__()
        
        assert gate_scheduler_type in ['linear', 'cosine', 'quadratic', 'cubic', 'sqrt'], (
            f"Unknown gate scheduler type: {gate_scheduler_type}. "
            "Must be one of ['linear', 'cosine', 'quadratic', 'cubic', 'sqrt']."
        )
        
        self.base_module = base_module
        self.gate_warmup_steps = gate_warmup_steps
        self.gate_scheduler_type = gate_scheduler_type
        self.training_step = 0
        
        # Determine in_features and out_features from the base module
        in_features = getattr(base_module, 'in_features', None)
        out_features = getattr(base_module, 'out_features', None)
        if in_features is None or out_features is None:
            raise ValueError(f"Cannot determine in_features or out_features from {base_module}.")
        
        # Initialize reference (ref) and transfer (tfr) LoRA modules
        self.lora = nn.ModuleDict({
            'ref': self._init_lora_module(in_features, out_features, 
                                          rank=ref_lora_config.r, 
                                          alpha=ref_lora_config.lora_alpha, 
                                          dropout=ref_lora_config.lora_dropout, 
                                          bias=False if ref_lora_config.bias == 'none' else True, 
                                          use_rslora=ref_lora_config.use_rslora),
            'tfr': self._init_lora_module(in_features, out_features, 
                                          rank=tfr_lora_config.r, 
                                          alpha=tfr_lora_config.lora_alpha, 
                                          dropout=tfr_lora_config.lora_dropout, 
                                          bias=False if tfr_lora_config.bias == 'none' else True, 
                                          use_rslora=tfr_lora_config.use_rslora),
        })
        
        # Initialize gate modules
        self.gate = self._init_lora_module(in_features, out_features, 
                                           rank=gate_rank, alpha=gate_alpha, 
                                           dropout=gate_dropout, bias=gate_bias, 
                                           use_rslora=gate_use_rslora)
        self.gate_act = nn.Sigmoid()


    def _init_lora_module(self, in_features, out_features, rank, alpha, dropout, bias, use_rslora):
        device = self.base_module.weight.device
        dropout_module = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        scaling = alpha / math.sqrt(rank) if use_rslora else alpha / rank
        
        # Initialize LoRA modules with extracted parameters
        lora_A = nn.Linear(in_features, rank, bias=bias, device=device)
        lora_B = nn.Linear(rank, out_features, bias=bias, device=device)
        
        # Initialize weights for the LoRA modules
        # nn.init.normal_(lora_A.weight, mean=0.0, std=1 / math.sqrt(rank))
        nn.init.kaiming_uniform_(lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(lora_B.weight)
        
        lora_module = nn.ModuleDict({
            'A': lora_A,
            'B': lora_B,
            'dropout': dropout_module,
        })
        lora_module.scaling = scaling
        lora_module.use_bias = bias
        return lora_module


    def load_lora_weights(self, state_dict, prefix, lora_type):
        assert lora_type in ['ref', 'tfr'], "Invalid LoRA type. Must be 'ref' or 'tfr'."
        
        device_A = self.lora[lora_type].A.weight.device # type: ignore
        device_B = self.lora[lora_type].B.weight.device # type: ignore

        key_A = f'{prefix}.lora_A.weight'
        key_B = f'{prefix}.lora_B.weight'

        with torch.no_grad():
            self.lora[lora_type].A.weight.copy_(state_dict[key_A].to(device_A)) # type: ignore
            self.lora[lora_type].B.weight.copy_(state_dict[key_B].to(device_B)) # type: ignore

            if self.lora[lora_type].use_bias:
                if f'{prefix}.lora_A.bias' in state_dict:
                    self.lora[lora_type].A.bias.copy_(state_dict[f'{prefix}.lora_A.bias'].to(device_A)) # type: ignore
                if f'{prefix}.lora_B.bias' in state_dict:
                    self.lora[lora_type].B.bias.copy_(state_dict[f'{prefix}.lora_B.bias'].to(device_B)) # type: ignore    


    def gate_scheduler(self):
        start = 0.0
        end = 1.0
        step = self.training_step
        warmup_steps = self.gate_warmup_steps
        scheduler_type = self.gate_scheduler_type
        
        if warmup_steps <= 0:
            return end
        
        t = min(max(step / warmup_steps, 0.0), 1.0)
        
        if scheduler_type == 'linear':
            p = t
        elif scheduler_type == 'cosine':
            # cosine warmup
            p = 0.5 * (1.0 - math.cos(math.pi * t))
        elif scheduler_type == 'quadratic':
            p = t**2
        elif scheduler_type == 'cubic':
            p = t**3
        elif scheduler_type == 'sqrt':
            p = math.sqrt(t)
        else:
            raise ValueError(
                f"Unknown gate scheduler type: {scheduler_type}. "
                "Must be one of ['linear', 'cosine', 'quadratic', 'cubic', 'sqrt']."
            )
        
        return start + (end - start) * p


    def forward(self, x):
        # Compute the base layer output
        base_out = self.base_module(x)
        
        # Check if the input tensor needs to be converted to LoRA module's dtype
        requires_conversion = not torch.is_autocast_enabled()
        if requires_conversion:
            # TODO: Add assertion to check if all weights have same dtype
            x = x.to(self.lora.tfr.A.weight.dtype) # pyright: ignore
        
        # Compute the outputs for the reference and transfer LoRA
        ref_lora_out = self.lora.ref.B(self.lora.ref.dropout(self.lora.ref.A(x))) * self.lora.ref.scaling # type: ignore
        tfr_lora_out = self.lora.tfr.B(self.lora.tfr.dropout(self.lora.tfr.A(x))) * self.lora.tfr.scaling # type: ignore
        
        # Compute the gate output and its complement
        gate_out = self.gate_act(self.gate.B(self.gate.dropout(self.gate.A(x)))) # type: ignore
        gate_out = gate_out * self.gate_scheduler() # Schedule gate output based on training step and warmup steps
        gate_comp_out = torch.ones(gate_out.shape, dtype=gate_out.dtype, device=gate_out.device) - gate_out
        
        # Compute the LoRA output
        lora_out = gate_out * ref_lora_out + gate_comp_out * tfr_lora_out
        
        # Check if the LoRA output needs to be converted back to the base layer's dtype
        if requires_conversion:
            lora_out = lora_out.to(base_out.dtype)
            
        return base_out + lora_out


    def set_training_step(self, training_step):
        self.training_step = training_step

In [ ]:
class LegamexLoraModel(nn.Module):
    def __init__(
            self, base_model, task_type, 
            
            # Reference (ref) and transfer (tfr) LoRA parameters
            ref_lora_config, tfr_lora_config, 
            
            # Gate parameters
            gate_warmup_steps, gate_scheduler_type='cosine', 
            gate_rank=1, gate_alpha=1, gate_dropout=0.0, 
            gate_bias=False, gate_use_rslora=False
        ):
        super().__init__()
        
        # Determine the base model name or path
        self.base_model_name_or_path = (
            ref_lora_config.base_model_name_or_path
            if hasattr(ref_lora_config, 'base_model_name_or_path')
            else base_model.config.name_or_path
        )
        if base_model.config.name_or_path != self.base_model_name_or_path:
            print("Using the base model name from the reference LoRA:", self.base_model_name_or_path)
        
        # Store the parameters
        self.base_model = base_model
        self.task_type = task_type
        self.ref_lora_config = ref_lora_config
        self.tfr_lora_config = tfr_lora_config
        self.gate_warmup_steps = gate_warmup_steps
        self.gate_scheduler_type = gate_scheduler_type
        self.gate_rank = gate_rank
        self.gate_alpha = gate_alpha
        self.gate_dropout = gate_dropout
        self.gate_bias = gate_bias
        self.gate_use_rslora = gate_use_rslora

        # Wrap the target modules with LegameX
        self.legamex_modules = {}
        self._wrap_target_modules_with_legamex()


    def _get_parent_module(self, module_name):
            parts = module_name.split('.')
            parent_module = self.base_model
            for part in parts[:-1]:
                parent_module = getattr(parent_module, part)
            return parent_module, parts[-1]


    def _wrap_target_modules_with_legamex(self):
        assert self.ref_lora_config.target_modules == self.tfr_lora_config.target_modules, \
            "Target modules for reference and transfer LoRA must be the same."
        target_modules = self.tfr_lora_config.target_modules
        
        for module_name, module in self.base_model.named_modules():
            # Skip the modules that are already wrapped with LegamexLora
            if isinstance(module, LegamexLora):
                module_name = module_name.replace('.', '__DOT__')
                self.legamex_modules[module_name] = module
                continue
            
            if any(target_module in module_name for target_module in target_modules):
                # Wrap the module with LegamexLora
                parent_module, child_name = self._get_parent_module(module_name)
                legamex_module = LegamexLora(
                    base_module=module,
                    ref_lora_config=self.ref_lora_config,
                    tfr_lora_config=self.tfr_lora_config,
                    gate_warmup_steps=self.gate_warmup_steps,
                    gate_scheduler_type=self.gate_scheduler_type,
                    gate_rank=self.gate_rank,
                    gate_alpha=self.gate_alpha,
                    gate_dropout=self.gate_dropout,
                    gate_bias=self.gate_bias,
                    gate_use_rslora=self.gate_use_rslora,
                )
                setattr(parent_module, child_name, legamex_module)
                
                # Store the wrapped module
                module_name = module_name.replace('.', '__DOT__')
                self.legamex_modules[module_name] = legamex_module
                
            # Store the qa_outputs modules if task_type is 'QUESTION_ANS'
            if isinstance(module, nn.Linear) and 'qa_outputs' in module_name and self.task_type == 'QUESTION_ANS':
                module_name = module_name.replace('.', '__DOT__')
                self.legamex_modules[module_name] = module
                
        # Freeze all parameters in the base model and unfreeze only the LegameX modules except for the reference LoRA
        self.freeze_all(verbose=False)
        self.unfreeze_legamex_except_ref_lora(verbose=True)


    def _json_safe(self, obj):
        if isinstance(obj, set):
            return list(obj)
        if isinstance(obj, dict):
            return {k: self._json_safe(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [self._json_safe(v) for v in obj]
        return obj


    def freeze_all(self, verbose=False):
        for p in self.base_model.parameters():
            p.requires_grad = False
        if verbose:
            print("All modules are frozen.")


    def unfreeze_legamex_except_ref_lora(self, verbose=False):
        for module_name, module in self.legamex_modules.items():
            if isinstance(module, LegamexLora):
                # Unfreeze the transfer LoRA weights and biases
                module.lora.tfr.A.weight.requires_grad = True # type: ignore
                module.lora.tfr.B.weight.requires_grad = True # type: ignore
                if module.lora.tfr.use_bias: # type: ignore
                    if hasattr(module.lora.tfr.A, 'bias') and module.lora.tfr.A.bias is not None: # type: ignore
                        module.lora.tfr.A.bias.requires_grad = True # type: ignore
                    if hasattr(module.lora.tfr.B, 'bias') and module.lora.tfr.B.bias is not None: # type: ignore
                        module.lora.tfr.B.bias.requires_grad = True # type: ignore
                
                # Unfreeze the gate weights and biases
                module.gate.A.weight.requires_grad = True # type: ignore
                module.gate.B.weight.requires_grad = True # type: ignore
                if module.gate.use_bias:
                    if hasattr(module.gate.A, 'bias') and module.gate.A.bias is not None: # type: ignore
                        module.gate.A.bias.requires_grad = True # type: ignore
                    if hasattr(module.gate.B, 'bias') and module.gate.B.bias is not None: # type: ignore
                        module.gate.B.bias.requires_grad = True # type: ignore

            # Unfreeze the qa_outputs weight and bias if task_type is 'QUESTION_ANS'
            elif isinstance(module, nn.Linear) and 'qa_outputs' in module_name and self.task_type == 'QUESTION_ANS':    
                module.weight.requires_grad = True
                if hasattr(module, 'bias') and module.bias is not None:
                        module.bias.requires_grad = True

        if verbose:
            trainable = sum(p.numel() for p in self.base_model.parameters() if p.requires_grad)
            total = sum(p.numel() for p in self.base_model.parameters())
            print(f"Trainable parameters: {trainable:,} / {total:,}")


    def load_lora_weights(self, lora_path, lora_type):
        assert lora_type in ['ref', 'tfr'], "Invalid LoRA type. Must be 'ref' or 'tfr'."
        
        if not os.path.exists(lora_path):
            raise FileNotFoundError(f"LoRA weights file not found: {lora_path}")
        
        if lora_path.endswith('.safetensors'):
            state_dict = load_file(lora_path)
        else:
            state_dict = torch.load(lora_path, map_location='cpu')
            
        processed_module_names = set()
        for key in state_dict.keys():
            key_prefix, key_rest = key.rsplit('model.', 1)
            module_name = key_rest.split('.lora', 1)[0]
            module_name_sanitized = module_name.replace('.', '__DOT__')
            
            if module_name_sanitized in processed_module_names:
                continue
            
            if module_name_sanitized in self.legamex_modules:
                prefix = key_prefix + 'model.' + module_name
                self.legamex_modules[module_name_sanitized].load_lora_weights(state_dict, prefix, lora_type)
            else:
                print("LoRA weights not found for module:", module_name)
                
            processed_module_names.add(module_name_sanitized)

        print(f"LoRA weights loaded from {lora_path} into model.")


    @classmethod
    def from_pretrained(cls, save_dir: str, **kwargs):
        # Load the LegameX configuration
        with open(os.path.join(save_dir, 'legamex_config.json'), 'r') as f:
            config = json.load(f)
            
        # Re-create the base model
        base_model_id = config['base_model_name_or_path']
        task_type = config['task_type']
        if task_type == 'QUESTION_ANS':
            base_model = AutoModelForQuestionAnswering.from_pretrained(base_model_id)
        elif task_type == 'FEATURE_EXTRACTION':
            base_model = AutoModelForMaskedLM.from_pretrained(base_model_id)
        else:
            raise ValueError(f"Unknown task type: {task_type}")
        
        # Load the reference and transfer LoRA configurations
        ref_lora_config = LoraConfig(**config['ref_lora_config'])
        tfr_lora_config = LoraConfig(**config['tfr_lora_config'])
        
        # Re-create the LegameX model
        model = cls(
            base_model=base_model,
            task_type=task_type,
            ref_lora_config=ref_lora_config,
            tfr_lora_config=tfr_lora_config,
            gate_warmup_steps=config['gate_warmup_steps'],
            gate_scheduler_type=config['gate_scheduler_type'],
            gate_rank=config['gate_rank'],
            gate_alpha=config['gate_alpha'],
            gate_dropout=config['gate_dropout'],
            gate_bias=config['gate_bias'],
            gate_use_rslora=config['gate_use_rslora'],
        )
        
        # Load the LegameX state dict
        legamex_state_dict = torch.load(os.path.join(save_dir, 'legamex.pt'), map_location='cpu')
        for k, v in legamex_state_dict.items():
            parts = k.split('.', 1) # ['module_name', '<rest_of_path>']
            if len(parts) == 2:
                module_name, param_name = parts
                module = model.legamex_modules.get(module_name)
                if module is not None:
                    param = dict(module.named_parameters()).get(param_name)
                    if param is not None:
                        param.data.copy_(v)
        
        return model


    def save_pretrained(self, save_dir):
        # Create save directories
        os.makedirs(save_dir, exist_ok=True)
        # tfr_dir = os.path.join(save_dir, 'tfr')
        # os.makedirs(tfr_dir, exist_ok=True)
        
        # Store the LegameX and transfer component state dicts
        state_dict = {}
        # tfr_state_dict = {}
        
        for module_name, module in self.legamex_modules.items():
            for param_name, param in module.named_parameters():
                # Skip the base_module parameters
                if 'base_module' in param_name:
                    continue
                
                # Store the rest of parameters in the LegameX state dict
                param_name_full = f'{module_name}.{param_name}'
                param_cpu = param.detach().cpu()
                state_dict[param_name_full] = param_cpu
                
                # # Store the transfer component and qa_outputs parameters separately
                # if 'lora.tfr' in param_name or 'qa_outputs' in module_name:
                #     param_name_full_sanitized = param_name_full.replace('__DOT__', '.').replace('lora.tfr.', 'lora_')
                #     tfr_param_cpu = param_cpu

                #     # If the parameter is a transfer component weight, apply the gate complement weight to it
                #     if 'lora.tfr' in param_name and 'weight' in param_name:
                #         gate_weight_name = param_name.replace('lora.tfr', 'gate')
                        
                #         try:
                #             gate_param = module.get_parameter(gate_weight_name)
                #             gate_weight = gate_param.detach().cpu()
                #             gate_comp_weight = 1.0 - gate_weight
                #             tfr_param_cpu = tfr_param_cpu * gate_comp_weight
                #         except AttributeError:
                #             raise AttributeError(
                #                 f"Could not find matching gate weight component '{gate_weight_name}' "
                #                 f"in module '{module_name}' to pair with transfer component weights."
                #             )
                        
                #     tfr_state_dict[param_name_full_sanitized] = tfr_param_cpu
                    
        torch.save(state_dict, os.path.join(save_dir, 'legamex.pt'))
        
        # Save the transfer component state dict and its configuration
        # save_file(tfr_state_dict, os.path.join(tfr_dir, 'adapter_model.safetensors'))
        # with open(os.path.join(tfr_dir, 'adapter_config.json'), 'w') as f:
        #     json.dump(self._json_safe(self.tfr_lora_config.to_dict()), f, indent=4)
        
        # Save the LegameX configuration for rebuilding wrapper
        config = {
            'base_model_name_or_path': self.base_model_name_or_path,
            'task_type': self.task_type,
            'ref_lora_config': self._json_safe(self.ref_lora_config.to_dict()),
            'tfr_lora_config': self._json_safe(self.tfr_lora_config.to_dict()),
            'gate_warmup_steps': self.gate_warmup_steps,
            'gate_scheduler_type': self.gate_scheduler_type,
            'gate_rank': self.gate_rank,
            'gate_alpha': self.gate_alpha,
            'gate_dropout': self.gate_dropout,
            'gate_bias': self.gate_bias,
            'gate_use_rslora': self.gate_use_rslora,
        }
        with open(os.path.join(save_dir, 'legamex_config.json'), 'w') as f:
            json.dump(config, f, indent=4)


    def merge_and_unload(self):
        self.eval() # Ensure dropout in the LoRA branches is disabled

        # Warn if any gate is not fully warmed up yet (gate_scheduler() < 1.0)
        cold_gates = [
            name for name, module in self.legamex_modules.items()
            if isinstance(module, LegamexLora) and module.gate_scheduler() < 1.0
        ]
        if cold_gates:
            print(
                f"Warning: {len(cold_gates)} gate(s) are not fully warmed up "
                f"(e.g. {cold_gates[0]}); the merge assumes gate_scheduler() == 1.0."
            )

        with torch.no_grad():
            for module_name, module in self.legamex_modules.items():
                # The qa_outputs head is already trained in the base model; nothing to merge
                if not isinstance(module, LegamexLora):
                    continue

                # Restore the real module name and locate its parent in the base model
                real_module_name = module_name.replace('__DOT__', '.')
                parent_module, child_name = self._get_parent_module(real_module_name)

                # Get the base module, reference LoRA, transfer LoRA, and gate components
                base_module = module.base_module
                ref = module.lora.ref
                tfr = module.lora.tfr
                gate = module.gate
                
                # Merge the reference and transfer LoRA biases if enabled
                if ref.use_bias: # type: ignore
                    assert base_module.bias is not None
                    raise NotImplementedError("Merging the reference LoRA bias is not implemented yet.")
                if tfr.use_bias: # type: ignore
                    assert base_module.bias is not None
                    raise NotImplementedError("Merging the transfer LoRA bias is not implemented yet.")
                
                # Compute the gate weights and their complement
                gate_W = torch.sigmoid(gate.B.weight @ gate.A.weight) # (out_features, in_features)
                gate_comp_W = 1.0 - gate_W                            # (out_features, in_features)

                # Compute the reference and transfer LoRA weights
                ref_W = (ref.B.weight @ ref.A.weight) * ref.scaling # (out_features, in_features), type: ignore
                tfr_W = (tfr.B.weight @ tfr.A.weight) * tfr.scaling # (out_features, in_features), type: ignore

                # Merge the reference and transfer LoRA weights into the base module
                delta_W = (ref_W * gate_W + tfr_W * gate_comp_W).to(base_module.weight.dtype)
                base_module.weight.data += delta_W

                # Unload: replace the wrapper with the merged base module
                setattr(parent_module, child_name, base_module)

        # The model is now fully merged; drop the wrapper registry
        self.legamex_modules = {}

        return self.base_model


    def forward(self, *args, **kwargs):
            kwargs.pop('num_items_in_batch', None)
            return self.base_model.forward(*args, **kwargs)


    def set_training_step(self, training_step):
        for module in self.legamex_modules.values():
            if hasattr(module, 'set_training_step'):
                module.set_training_step(training_step)


    def __getattr__(self, name):
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self.base_model, name)

In [13]:
# Initialize the LegameX model
legamex_model = LegamexLoraModel.from_pretrained(str(legamex_dir))
legamex_model = legamex_model.to(DEVICE)
print("device:", legamex_model.device)

Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable parameters: 1,494,530 / 280,274,690
device: cuda:0


In [14]:
# Sanity check
for n, p in legamex_model.named_parameters():
    if "layer.0.attention.output.dense.lora.ref" not in n:
        continue
    check_parameter(n, p)
    print()

name    : base_model.roberta.encoder.layer.0.attention.output.dense.lora.ref.A.weight
device  : cuda:0
dtype   : torch.float32
norm    : 1.7452431917190552

name    : base_model.roberta.encoder.layer.0.attention.output.dense.lora.ref.B.weight
device  : cuda:0
dtype   : torch.float32
norm    : 0.2991487979888916



In [ ]:
# Sanity check: merge the LegameX deltas into the base model and unload the wrappers
legamex_model = legamex_model.eval()

# Forward pass with the gated (unmerged) model first, for comparison
dummy_inputs = tokenizer("What is the capital of France?", return_tensors="pt").to(DEVICE)
with torch.no_grad():
    gated_outputs = legamex_model(**dummy_inputs)

merged_model = legamex_model.merge_and_unload()

# Verify that all LegamexLora wrappers have been unloaded
remaining_wrappers = [n for n, m in merged_model.named_modules() if isinstance(m, LegamexLora)]
print("Remaining LegamexLora wrappers:", len(remaining_wrappers))

# Forward pass with the merged model
with torch.no_grad():
    merged_outputs = merged_model(**dummy_inputs)

# Compare the gated and merged logits (a small diff indicates a good merge approximation)
for head in ["start_logits", "end_logits"]:
    diff = (getattr(gated_outputs, head) - getattr(merged_outputs, head)).abs()
    print(f"{head:<13}: max abs diff = {diff.max().item():.6f}")

# Compare weight norms: pristine base model (loaded in the Model section) vs merged model
print()
for n, p in merged_model.named_parameters():
    if "layer.0.attention.output.dense.weight" not in n:
        continue
    base_p = dict(base_model.named_parameters())[n]
    check_parameter(n, base_p)
    print()
    check_parameter(n, p)
    break

print("Base   qa_outputs weight norm:", base_model.qa_outputs.weight.norm().item())
print("Merged qa_outputs weight norm:", merged_model.qa_outputs.weight.norm().item())

Remaining LegamexLora wrappers: 0
start_logits : max abs diff = 6.115014
end_logits   : max abs diff = 5.627743

name    : roberta.encoder.layer.0.attention.output.dense.weight
device  : cpu
dtype   : torch.float32
norm    : 28.121917724609375

name    : roberta.encoder.layer.0.attention.output.dense.weight
device  : cuda:0
dtype   : torch.float32
norm    : 28.125778198242188
Base   qa_outputs weight norm: 0.7894015312194824
Merged qa_outputs weight norm: 1.1798607110977173


# Merging and Uploading

In [ ]:
legamex_model = legamex_model.eval() # Good practice
merged_model = legamex_model.merge_and_unload()

# Upload the merged model to Hugging Face
merged_model.push_to_hub(hub_merged_model_id)
tokenizer.push_to_hub(hub_merged_model_id)

print(f"Merged model uploaded to: https://huggingface.co/{hub_merged_model_id}")